# 05 — Phase 4: GQA + PagedAttention + KIVI (All Combined)

三层优化全部启用：
1. **架构层 (GQA)**: 2 KV heads — 从模型设计源头减轻 KV Cache 负荷
2. **系统层 (PagedAttention)**: block-based 内存管理 — 消除碎片
3. **数值层 (KIVI 2-bit)**: 旧 block 量化 — 压缩显存占用

**预期**: 最低内存占用 + 最低碎片率 + 最长可处理上下文

这是在 Jetson Orin NX 10-12 GB 统一内存下，处理「长期医疗推理任务」的最终方案。

In [ ]:
import sys, gc, torch
sys.path.insert(0, '..')

from src.metrics import (
    measure_generation, run_benchmark, find_oom_threshold,
    print_memory_budget, JETSON_USABLE_GB,
)
from src.paged_kivi_cache import PagedKIVICache
from src.perplexity import compute_ppl_with_kv_cache
from src.dataset_utils import load_prompts

In [ ]:

from src.jetson_utils import load_model_safe, aggressive_cleanup

MODEL_NAME = "Qwen/Qwen2.5-1.5B-Instruct"
FALLBACK_NAME = "Qwen/Qwen2.5-0.5B-Instruct"
DEVICE = "cuda"

model, tokenizer = load_model_safe(
    MODEL_NAME,
    fallback_name=FALLBACK_NAME,
    device=DEVICE,
)
budget = print_memory_budget(model, DEVICE)


In [ ]:
# Combined 参数
BLOCK_SIZE = 16
RESIDUAL_BLOCKS = 8   # 8 blocks × 16 tokens = 128 tokens FP16
BITS = 2
GROUP_SIZE = 32

prompts = load_prompts('../results/pubmedqa_prompts.json')
print(f"Loaded {len(prompts)} prompts")
print(f"\nPagedKIVICache(block={BLOCK_SIZE}, residual_blocks={RESIDUAL_BLOCKS}, bits={BITS})")

In [ ]:
# OOM 探测
print("Combined OOM 探测...")
oom_all = find_oom_threshold(
    model, tokenizer,
    context_lengths=[256, 512, 1024, 2048, 4096, 8192, 16384],
    max_new_tokens=32,
    cache_factory=lambda: PagedKIVICache(
        block_size=BLOCK_SIZE, residual_blocks=RESIDUAL_BLOCKS,
        bits=BITS, group_size=GROUP_SIZE,
    ),
)

print(f"最大安全长度: {oom_all['max_safe_length']}")
print(f"OOM 发生在  : {oom_all['oom_length']}")
for r in oom_all['results']:
    if r['status'] == 'ok':
        print(f"  ctx={r['context_length']:>6}  peak={r['peak_memory_mb']:>7.0f} MB  "
              f"util={r['utilization']*100:>5.1f}%")
    else:
        print(f"  ctx={r['context_length']:>6}  → {r['status']}")

In [ ]:
# 单样本测试
cache = PagedKIVICache(BLOCK_SIZE, RESIDUAL_BLOCKS, BITS, GROUP_SIZE)
m = measure_generation(model, tokenizer, prompts[0]['prompt'],
                       max_new_tokens=64, cache_impl=cache)

print(f"TTFT           : {m.ttft_ms:.1f} ms")
print(f"TPOT           : {m.tpot_ms:.1f} ms")
print(f"KV Cache       : {m.kv_cache_memory_mb:.1f} MB")
print(f"Peak memory    : {m.peak_memory_mb:.0f} MB")
print(f"Memory util    : {m.memory_utilization*100:.1f}%")
print(f"Fragmentation  : {m.memory_fragmentation:.3f}")
print(f"\nCache: {cache}")

In [ ]:
# 完整 benchmark
results_all = run_benchmark(
    model, tokenizer, prompts,
    cache_factory=lambda: PagedKIVICache(
        BLOCK_SIZE, RESIDUAL_BLOCKS, BITS, GROUP_SIZE,
    ),
    max_new_tokens=256,
    warmup_runs=2, num_runs=3,
)
print(f"Completed {len(results_all)} samples")

In [ ]:
import pandas as pd

df = pd.DataFrame(results_all)
df['config'] = 'GQA+Paged+KIVI'
df.to_csv('../results/all_combined.csv', index=False)

print(df[['ttft_ms', 'tpot_ms', 'peak_memory_mb',
           'kv_cache_memory_mb', 'memory_fragmentation',
           'memory_utilization']].describe().round(2))

In [ ]:
# PPL 退化评估
eval_texts = [p['reference_answer'] for p in prompts[:20] if p.get('reference_answer')]

ppl_combined = compute_ppl_with_kv_cache(
    model, tokenizer, eval_texts,
    cache_factory=lambda: PagedKIVICache(
        BLOCK_SIZE, RESIDUAL_BLOCKS, BITS, GROUP_SIZE,
    ),
    max_length=512,
)

import json
with open('../results/ppl_results.json') as f:
    ppl_all = json.load(f)

baseline_ppl = ppl_all['GQA_only']['ppl']
combined_ppl = ppl_combined['ppl']
degradation = (combined_ppl - baseline_ppl) / baseline_ppl * 100

print(f"Baseline PPL      : {baseline_ppl:.2f}")
print(f"Paged+KIVI PPL    : {combined_ppl:.2f}")
print(f"退化              : {degradation:+.1f}%")

ppl_all['GQA+Paged+KIVI'] = ppl_combined
with open('../results/ppl_results.json', 'w') as f:
    json.dump(ppl_all, f, indent=2)

In [ ]:
del model
gc.collect()
torch.cuda.empty_cache()
print(f"GPU memory after cleanup: {torch.cuda.memory_allocated() / (1024**2):.0f} MB")